# Olist E-Commerce: load data, build the sales view, RFM segmentation

Run the cells **top to bottom**. Prerequisites (see the README):
1. MySQL is running and `sql/db_create.sql` has been executed (creates the `olist_ecommerce` database).
2. `.env` exists (copy `.env.example`) with `DB_PASSWORD` set, and the nine Olist CSVs are in `OLIST_DATA_DIR`.
3. `pip install -r requirements.txt`, then start Jupyter from the repo root or from `python/`.

Steps: (1) load CSVs into MySQL, (2) create `vw_master_sales`, (3) extract item-level rows, (4) data-quality checks, (5) RFM + segments, (6) write `customer_segments`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

# Make python/config.py and python/rfm.py importable whether Jupyter was started in the repo root or in python/
_here = Path.cwd()
sys.path.insert(0, str(_here if (_here / "rfm.py").exists() else _here / "python"))

import config
import rfm

engine = create_engine(config.get_db_url())  # raises a clear error if DB_PASSWORD is missing
REPO_ROOT = config.REPO_ROOT
print("Data folder:", config.get_data_dir())

Data folder: E:\Ecommerce-Projects\Ecommerce-Sales-Analytics-git\data\raw


## 1. Load the Olist CSVs into MySQL
Re-running replaces the tables. Tables are named after the files, e.g. `olist_orders_dataset.csv` -> `orders`.

In [2]:
data_dir = config.check_data_dir()  # stops with a clear message if the folder or any CSV is missing

failed = []
for file in config.EXPECTED_CSV_FILES:
    table_name = config.table_name_for(file)
    try:
        df_csv = pd.read_csv(data_dir / file)
        print(f"Loading {file} -> {table_name} ({len(df_csv):,} rows)...")
        df_csv.to_sql(name=table_name, con=engine, if_exists="replace", index=False)
    except Exception as e:
        failed.append((file, str(e)))
        print(f"  ERROR processing {file}: {e}")

if failed:
    raise RuntimeError(f"{len(failed)} file(s) failed to load: {[f for f, _ in failed]}")
print("All files loaded.")

Loading olist_customers_dataset.csv -> customers (99,441 rows)...
Loading olist_geolocation_dataset.csv -> geolocation (1,000,163 rows)...
Loading olist_order_items_dataset.csv -> order_items (112,650 rows)...
Loading olist_order_payments_dataset.csv -> order_payments (103,886 rows)...
Loading olist_order_reviews_dataset.csv -> order_reviews (99,224 rows)...
Loading olist_orders_dataset.csv -> orders (99,441 rows)...
Loading olist_products_dataset.csv -> products (32,951 rows)...
Loading olist_sellers_dataset.csv -> sellers (3,095 rows)...
Loading product_category_name_translation.csv -> product_category_name_translation (71 rows)...
All files loaded.


## 2. Create the sales view
Executes `sql/fact_table.sql` (`vw_master_sales`, one row per **order item**, delivered orders only).

In [3]:
view_sql = (REPO_ROOT / "sql" / "fact_table.sql").read_text(encoding="utf-8")
with engine.begin() as conn:
    conn.exec_driver_sql(view_sql)
print("vw_master_sales created.")

vw_master_sales created.


## 3. Extract item-level rows for RFM

In [4]:
query = """
SELECT order_id, order_item_id, customer_unique_id, purchase_date, product_price, delivery_cost
FROM vw_master_sales
"""
df = pd.read_sql(query, engine)
df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce")
print(df.head())
print(f"Item-level rows: {len(df):,} | distinct orders: {df['order_id'].nunique():,}")

                           order_id  order_item_id  \
0  01835ef41c5ac66dd9ec6084fa1d0533              1   
1  04a59fe46fc5aa341da5378c195ed53c              1   
2  062d3e3b803bd3e569cf8c165602f8f4              1   
3  06f3401b97e4d89032cbd5ac7670247c              1   
4  09bb35a3939c3181b4a5619cb3918976              1   

                 customer_unique_id purchase_date  product_price  \
0  f64ee7b44a9c2170a0db74d6bc3d1ba8    2017-05-11          56.99   
1  a3d335d8dd759126b58367c6053bb20a    2017-11-26          60.00   
2  b26517697abc966b9d0547f53ccccbe6    2018-06-25         122.99   
3  4211dcafb715e60bb929a44d3e4951a2    2018-04-26         148.00   
4  99ecf7e96d410a094bc95f5d3cb9d98f    2017-11-26          99.00   

   delivery_cost  
0          15.15  
1          15.18  
2          63.11  
3          23.62  
4          17.94  
Item-level rows: 110,197 | distinct orders: 96,478


## 4. Data-quality checks
Fails fast on missing keys, unparseable dates, negative values or duplicate order-item keys.

In [5]:
quality = rfm.data_quality_report(df)
for key, value in quality.items():
    print(f"{key:40s} {value}")
rfm.assert_data_quality(quality)

rows                                     110197
distinct_orders                          96478
distinct_customers                       93358
critical_missing_keys                    0
critical_invalid_or_missing_dates        0
critical_negative_values                 0
missing_price_or_freight                 0
orders_with_multiple_customers           0
orders_with_multiple_dates               0
critical_duplicate_order_item_keys       0
date_min                                 2016-09-15
date_max                                 2018-08-29


## 5. RFM and segmentation
* **Recency**: days from the customer's last purchase to (latest purchase date + 1 day).
* **Frequency**: number of **distinct orders** per customer (not item rows).
* **Monetary**: sum of product price + freight.
* R and M are scored 1-4 by quartile; F is 1, 2, 3, or 4 (4+ orders).
* A customer with only one distinct order is never a Champion or Loyal Customer. Full rules: `rfm.assign_segment`.

In [6]:
rfm_table = rfm.build_rfm(df)

print(rfm_table.head())
print("\nRFM summary statistics:")
print(rfm_table[["Recency", "Frequency", "Monetary"]].describe())
print("\nCustomers per segment:")
print(rfm_table["Customer_Segment"].value_counts())

# Sanity check: no one-time buyer may be Champions / Loyal Customers
assert not (
    (rfm_table["Frequency"] == 1) & rfm_table["Customer_Segment"].isin(["Champions", "Loyal Customers"])
).any()

                 customer_unique_id  Recency  Frequency  Monetary  R_Score  \
0  0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90        4   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19        4   
2  0000f46a3911fa3c0805444483337064      538          1     86.22        1   
3  0000f6ccb0745a6a4b88665a16c9f078      322          1     43.62        2   
4  0004aac84e0df4da2b147fca70cf8255      289          1    196.89        2   

   M_Score  F_Score RFM_Concat  RFM_Total_Score Customer_Segment  
0        3        1        413                8    New Customers  
1        1        1        411                6    New Customers  
2        2        1        112                4  Lost / One-Time  
3        1        1        211                4  Lost / One-Time  
4        4        1        214                7  Lost / One-Time  

RFM summary statistics:
            Recency     Frequency      Monetary
count  93358.000000  93358.000000  93358.000000
mean    

## 6. Save segments to MySQL (used by the Power BI report)

In [7]:
rfm_table.to_sql(name="customer_segments", con=engine, if_exists="replace", index=False)
print(f"customer_segments written ({len(rfm_table):,} customers).")

customer_segments written (93,358 customers).
